In [1]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install libraries
!apt-get update && apt-get install -y libclang-dev
!pip install tokenizers==0.13.3
!pip install transformers==4.28.0 torch torchaudio numpy tqdm

Mounted at /content/drive
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,302 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,624 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:13 https://p

In [2]:
import os

# Paths
zip_path = "/content/drive/MyDrive/Plagiarism-Detection-System/data/processed_smp.zip"
extract_path = "/content/data"

# Decompression
if not os.path.exists(extract_path):
    print("Unzipping dataset...")
    !unzip -q "{zip_path}" -d "{extract_path}"
    print("Done!")
else:
    print("Files already extracted.")

Unzipping dataset...
Done!


In [ ]:
import os
import torch
import soundfile as sf
import torchaudio.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Any

class AudioDataset(Dataset):
    def __init__(self, tracks_dir: str, audio_processor=None):
        self.tracks_dir = tracks_dir
        self.tracklist = sorted([
            t for t in os.listdir(tracks_dir)
            if os.path.isdir(os.path.join(tracks_dir, t)) and not t.startswith('.')
        ])
        self.audio_processor = audio_processor
        self.sample_rate = None

    def __len__(self) -> int:
        return len(self.tracklist)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        audios = {}
        track_name = self.tracklist[idx]
        track_path = os.path.join(self.tracks_dir, track_name)

        versions = sorted([
            v for v in os.listdir(track_path)
            if os.path.isdir(os.path.join(track_path, v)) and not v.startswith('.')
        ])

        for version in versions:
            version_path = os.path.join(track_path, version)
            files = sorted([f for f in os.listdir(version_path) if f.endswith('.wav')])

            if not files: continue

            audios[version] = []
            for file in files:
                file_path = os.path.join(version_path, file)

                data, samplerate = sf.read(file_path)
                self.sample_rate = samplerate

                waveform = torch.from_numpy(data).float()
                if waveform.ndim == 1:
                    waveform = waveform.unsqueeze(0)
                else:
                    waveform = waveform.t()

                audios[version].append(waveform)

            # MERT Preprocessing
            if self.audio_processor:
                target_sr = self.audio_processor.sampling_rate
                processed_audios = []
                for waveform in audios[version]:
                    if self.sample_rate is not None and self.sample_rate != target_sr:
                        waveform = F.resample(waveform, int(self.sample_rate), target_sr)

                    inputs = self.audio_processor(
                        waveform.squeeze().numpy(),
                        sampling_rate=target_sr,
                        return_tensors="pt"
                    )["input_values"].squeeze()
                    processed_audios.append(inputs)

                audios[version] = processed_audios

        if audios:
            min_frames = min([len(audios[v]) for v in audios.keys()])
            for v in audios.keys():
                audios[v] = audios[v][:min_frames]

        return {'track': track_name, 'audios': audios}

def audio_collate_fn(batch):
    batch_dict = {}
    max_len = 0
    for item in batch:
        for version in item["audios"]:
            for segment in item["audios"][version]:
                if segment.shape[-1] > max_len: max_len = segment.shape[-1]

    for item in batch:
        track_name = item["track"]
        batch_dict[track_name] = []
        for version in item["audios"]:
            padded_segments = []
            for segment in item["audios"][version]:
                pad_amount = max_len - segment.shape[-1]
                padded_seg = torch.nn.functional.pad(segment, (0, pad_amount))
                padded_segments.append(padded_seg)
            batch_dict[track_name].append(torch.stack(padded_segments))
    return batch_dict

def create_audio_dataloader(tracks_dir, batch_size=1, num_workers=2, audio_processor=None):
    dataset = AudioDataset(tracks_dir, audio_processor=audio_processor)
    return DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False, collate_fn=audio_collate_fn)

In [4]:
!pip install soundfile

In [6]:
import numpy as np
import torch
import torchaudio
from transformers import AutoModel, Wav2Vec2FeatureExtractor
from tqdm import tqdm
import os

try:
    import soundfile
    torchaudio.set_audio_backend("soundfile")
    print("Audio backend set to: soundfile")
except:
    print("Could not set soundfile backend explicitly. Hoping for auto-detection.")

INPUT_DIR = "/content/data/processed_smp"
OUTPUT_DIR = "/content/drive/MyDrive/Plagiarism-Detection-System/data/mert_embeddings"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def run_mert_extraction():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load Model
    print("Loading MERT model...")
    processor = Wav2Vec2FeatureExtractor.from_pretrained("m-a-p/MERT-v1-95M")
    model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True).to(device)
    model.eval()

    # Time reduction layer
    time_reduce = torch.nn.AvgPool1d(kernel_size=10, stride=10, count_include_pad=False)

    # Dataloader
    dataloader = create_audio_dataloader(INPUT_DIR, batch_size=1, audio_processor=processor)

    print(f"Starting extraction for {len(dataloader)} tracks...")

    with torch.no_grad():
        for batch in tqdm(dataloader):
            for track_name in batch.keys():

                versions_list = batch[track_name]
                if len(versions_list) < 2:
                    print(f"Skipping {track_name}: Found less than 2 versions.")
                    continue

                pair_folder = os.path.join(OUTPUT_DIR, f"{track_name}")
                os.makedirs(pair_folder, exist_ok=True)

                if os.path.exists(os.path.join(pair_folder, "original.npy")) and \
                   os.path.exists(os.path.join(pair_folder, "cover.npy")):
                    continue

                processed_versions = []

                for version_segments in versions_list:
                    version_segments = version_segments.to(device)

                    # Pass through MERT
                    hidden_states = model(version_segments, output_hidden_states=True).hidden_states

                    # Select Layers (2, 5, 8, 11) & Reduce Time Dimension
                    features = torch.stack(
                        [time_reduce(h.detach()[:, :, :].permute(0,2,1)).permute(0,2,1) for h in hidden_states[2::3]],
                        dim=1
                    )
                    # Squeeze if batch size 1 added extra dim, ensure shape (Layers, Time, Dim)
                    features = features.squeeze(0).cpu().numpy()
                    processed_versions.append(features)


                np.save(os.path.join(pair_folder, "cover.npy"), processed_versions[0])
                np.save(os.path.join(pair_folder, "original.npy"), processed_versions[1])

    print(f"\n✅ Extraction Complete! Saved to: {OUTPUT_DIR}")

run_mert_extraction()

Could not set soundfile backend explicitly. Hoping for auto-detection.
Using device: cuda
Loading MERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

...

Skipping pair_62: Found less than 2 versions.
Skipping pair_63: Found less than 2 versions.


 89%|████████▉ | 141/158 [07:17<00:55,  3.29s/it]

Skipping pair_84: Found less than 2 versions.


100%|██████████| 158/158 [08:08<00:00,  3.09s/it]


✅ Extraction Complete! Saved to: /content/drive/MyDrive/Plagiarism-Detection-System/data/mert_embeddings
